# Earthquake analysis on M1M3 using a VMS data

This is the new notebook for earthquake analysis after SITCOM-1399 and SITCOM-1799, mainly to provide plots to SPIE proceedings (but also can be used further) to compare before and after the software limit changes of M1M3 reaction to the earthquake.

* This analysis requires VMS file, and in a "parquet" form to reduce file size.

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import os.path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import h5py
import numpy as np

from astropy.time import Time
from astropy.time import TimeDelta
from astropy import units as u
from astropy import constants as const

import matplotlib.dates as mdates
from matplotlib.ticker import FormatStrFormatter
from matplotlib.dates import DateFormatter
import matplotlib.dates as mdates
from matplotlib import ticker

from scipy.signal import stft, butter, filtfilt
from scipy.signal import find_peaks_cwt
from scipy import signal
from scipy.fft import fft, rfft, fftfreq, rfftfreq

from lsst_efd_client import EfdClient
from lsst.summit.utils.tmaUtils import TMAEventMaker, TMAState, TMAEvent
from lsst.summit.utils.efdUtils import getEfdData, makeEfdClient
from lsst.ts.xml.enums import MTM1M3, MTMount
import read_hdf

In [ ]:
client = EfdClient("usdf_efd")

In [ ]:
date_dict = {
    "date": "2026-03-09",
    "start_time": "17:12:00.00",
    "end_time": "17:14:00.00",
}
date = date_dict["date"]
start_time = Time(f"{date} {date_dict['start_time']}")
end_time = Time(f"{date} {date_dict['end_time']}")

In [ ]:
start_time

In [ ]:
df_mtm1m3_dst = getEfdData(
    client, "lsst.sal.MTM1M3.logevent_detailedState", begin=start_time, end=end_time
)

In [ ]:
df_mtm1m3_dst

In [ ]:
def vms_data_to_pandas(filename, vms_type, begin_time=None, end_time=None):
    """
    Converts VMS data in the given HDF5 file to a Pandas DataFrame.

    Args:
    filename: Path to the HDF5 file containing the VMS data.
    vms_type: The type of VMS data in the file. Must be "m1m3", "m2", or
      "rotator".
    begin_time: The start time of the data to include in the DataFrame. If None,
      all data will be included.
    end_time: The end time of the data to include in the DataFrame. If None, all
      data will be included.

    Returns:
    A Pandas DataFrame containing the VMS data.
    """
    if vms_type == "m1m3":
        key_dict = key_m1m3_dict
    elif vms_type == "m2":
        key_dict = key_m2_dict
    elif vms_type == "rotator":
        key_dict = key_rotator_dict
    else:
        raise ValueError("vms_type must be m1m3,m2, or rotator")

    f = h5py.File(filename, "r")
    times = f["timestamp"][::1]
    dkeys = "XYZ"

    data_dict = {}
    if (begin_time is not None) & (end_time is not None):
        sel = (times > begin_time) & (times < end_time)
    else:
        sel = np.ones(times.size).astype(bool)
    data_dict["times"] = times[sel]
    for key in key_dict.keys():
        # multiply values stored in hdf5 files by 2 in order to convert the acceleration values to mg
        data_dict[key_dict[key]] = f[key][::1][sel] * 2.0
    data_frame = pd.DataFrame(data_dict)
    for j in np.arange(int(len(key_dict) / 3)) + 1:
        data_frame[f"total_{j}"] = np.linalg.norm(
            data_frame[[f"{vms_type}_{i}_{j}" for i in ["x", "y", "z"]]].values, axis=1
        )

    return data_frame

In [ ]:
system = "M1M3"
vms_date = "2026-03-09"
vms_top_dir = "/home/h/hyeyun/vmsdata"
year, month = vms_date.split("-")[0:2]

# Directory containing VMS data
vms_dir = os.path.join(vms_top_dir, year, month)

# Check if a parquet file exists
vms_mx_parquet_filename = os.path.join(vms_dir, f"{system}-" + vms_date + "T00:00.parquet")
if os.path.isfile(vms_mx_parquet_filename):
    print(f"Reading VMS data from parquet file:{vms_mx_parquet_filename}")
    vms_m1m3_data = pd.read_parquet(vms_mx_parquet_filename)
else:
    print("Parquet file not found")

# Try to avoid that corrections in the next cell to be applied more than 1 time
applied_corrections_m1m3 = False

# Check that the corrections has not been applied already
if applied_corrections_m1m3 == False:

    # Subtract 37s to timestamps in order to take into account the difference between the TAI and UTC
    vms_m1m3_data.times = vms_m1m3_data.times - pd.Timedelta(value=37, unit="seconds")
    
    # Convert acceleration from milli-g to m/s²
    #for sensor in range(3):
    #    for axis in "xyz":
    #        key = f"{system.lower()}_{axis}_{sensor+1}"
    #        vms_mx_data[key] = 1e-3 * 9.8 * vms_mx_data[key]

    applied_corrections_m1m3 = True
else:
    print("Corrections have already been applied for M1M3 - Skipped")

In [ ]:
# Create a datetime index
vms_m1m3_data = vms_m1m3_data.set_index('times')

In [ ]:
start = pd.Timestamp(start_time.unix, unit="s")
end = pd.Timestamp(end_time.unix, unit="s") + pd.Timedelta(value=60, unit="seconds")
print(start, end)

In [ ]:
cut = (vms_m1m3_data.index >= start) & (vms_m1m3_data.index < end)

In [ ]:
start_ext = pd.Timestamp(start_time.unix, unit="s") - pd.Timedelta(value=60, unit="seconds")
end_ext = pd.Timestamp(end_time.unix, unit="s") + pd.Timedelta(value=3, unit="hours")
print(start_ext, end_ext)

cut_ext = (vms_m1m3_data.index >= start_ext) & (vms_m1m3_data.index < end_ext)

In [ ]:
t1 = Time("2026-03-09 06:09:00.000")
t2 = Time("2026-03-09 06:13:00.000")

In [ ]:
df_mtm1m3_af = getEfdData(
    client, "lsst.sal.MTM1M3.appliedForces", 
    columns = ["fx", "fy", "fz", "mx", "my", "mz"],
    begin=t1, end=t2
)
df_mtm1m3_ims = getEfdData(
    client, "lsst.sal.MTM1M3.imsData",
    begin=t1, end=t2
)

In [ ]:
t_start = pd.Timestamp(t1.unix, unit="s") 
t_end = pd.Timestamp(t2.unix, unit="s") 

cut = (vms_m1m3_data.index >= t_start) & (vms_m1m3_data.index < t_end)

In [ ]:
df_mtm1m3_af

In [ ]:
ax2 = [None]*3
fig, ax = plt.subplots(3, 1, sharex=True, dpi=125, figsize=(14, 14))
for i, axis in enumerate("xyz"):
    baseline = np.mean(df_mtm1m3_af[f"f{axis}"])
    ax[i].plot(df_mtm1m3_af.index, df_mtm1m3_af[f"f{axis}"]-baseline, lw=1)
    ax2[i] = ax[i].twinx()
    ax2[i].plot(vms_m1m3_data.index[cut], vms_m1m3_data[cut][f"m1m3_{axis}_1"], lw=0.1, c="red")
    ax[i].set_ylabel(f"Sum of f{axis} (N)")
    ax2[i].set_ylabel(f"VMS accel. {axis} (milli-g)", color="red")
    
ax[2].set_xlabel("Time")
        
fig.suptitle(f"Earthquake {date}")
fig.tight_layout()

In [ ]:
vms_m1m3_data[cut]

In [ ]:
df_mtm1m3_ims

In [ ]:
ax2 = [None]*3
fig, ax = plt.subplots(3, 2, sharex=True, dpi=125, figsize=(14, 14))
for i, axis in enumerate("xyz"):
    base_pos = np.mean(df_mtm1m3_ims[f"{axis}Position"])
    ax[i][0].plot(df_mtm1m3_ims.index, (df_mtm1m3_ims[f"{axis}Position"]-base_pos)*1.e6, lw=1)
    ax[i][0].set_ylabel(f"{axis}Position (microns)")
    base_rot = np.mean(df_mtm1m3_ims[f"{axis}Rotation"])
    ax[i][1].plot(df_mtm1m3_ims.index, (df_mtm1m3_ims[f"{axis}Rotation"]-base_rot)*3600, lw=1)
    ax[i][1].set_ylabel(f"{axis}Rotation (arcsec)")
    #ax[i][0].set_ylim([-200, 200])
    #ax[i][1].set_ylim([-10, 10])

for j in range(2):
    ax[2][j].xaxis.set_major_formatter(DateFormatter("%H-%M-%S"))
    plt.setp(ax[2][j].get_xticklabels(), rotation=45)
        
fig.suptitle(f"Earthquake {date}")
fig.tight_layout()